# 面试问题：Split Conformal Prediction 怎样把点预测校准成有有限样本覆盖保证的区间？

        ## 可直接复述的回答主线

        1. 点模型只给 ETA 预测，固定正负几分钟没有覆盖率依据，面对异方差样本容易过窄。
2. Split Conformal 把数据分成训练、校准和测试；训练点模型后只用独立校准集计算绝对残差。
3. 有限样本分位数使用 ceil((n+1)*(1-alpha)) 的保守顺序统计量，而不是随意取均值残差。
4. 新样本区间是 [prediction-q, prediction+q]，覆盖是边际保证，区间宽度直接反映校准误差。
5. 不能用训练残差校准，因为模型已拟合这些样本，会得到过窄区间和虚假高置信。
6. 生产还需检查 exchangeability、分群覆盖、分布漂移、在线重校准、截断边界和 abstention。

        后续实验会在同一批输入上依次展示朴素基线、手写核心机制、中间过程、失败修正和生产边界。

## 1. 真实案例与输入预览

案例是配送 ETA 回归：字段包括距离、是否下雨、骑手负载和实际分钟数。12 条训练、10 条独立校准、10 条测试均由确定性业务公式加可读噪声构造；受控小数据只解释机制，不代表线上覆盖。

In [1]:
import math  # 计算有限样本 conformal rank 的向上取整。
import numpy as np  # 手写线性回归、残差分位数和区间覆盖。
def make_records(prefix, specs):  # 根据业务特征和确定性扰动构造 ETA 记录。
    records = []  # 保存具有可读字段的配送样本。
    for index, (distance, rain, load, noise) in enumerate(specs, start=1):  # 逐规格生成实际 ETA。
        expected_minutes = 15.0 + 2.0 * distance + 7.0 * rain + 4.0 * load  # 定义脱敏教学中的基础配送规律。
        actual_minutes = expected_minutes + noise  # 加入交通、等餐等未建模扰动。
        records.append({"id": f"{prefix}-{index:02d}", "distance_km": distance, "rain": rain, "courier_load": load, "noise": noise, "actual_minutes": actual_minutes})  # 保存当前配送业务记录。
    return records  # 返回完整数据切分。
training_specs = [(1.0, 0, 0.2, -1.0), (2.0, 0, 0.3, 1.0), (3.0, 1, 0.4, -0.5), (4.0, 0, 0.6, 0.5), (5.0, 1, 0.7, -1.0), (6.0, 0, 0.2, 1.0), (7.0, 1, 0.9, 0.0), (8.0, 0, 0.5, -0.5), (9.0, 1, 0.8, 0.8), (10.0, 0, 0.4, -0.8), (11.0, 1, 0.6, 0.6), (12.0, 0, 0.7, 0.0)]  # 定义十二条训练样本的小扰动。
calibration_specs = [(1.5, 0, 0.3, -1.0), (2.5, 1, 0.5, 1.0), (3.5, 0, 0.7, -2.0), (4.5, 1, 0.2, 2.0), (5.5, 0, 0.8, -3.0), (6.5, 1, 0.4, 3.0), (7.5, 0, 0.6, -5.0), (8.5, 1, 0.9, 5.0), (9.5, 0, 0.3, -7.0), (10.5, 1, 0.7, 7.0)]  # 定义十条独立校准样本的宽扰动。
test_specs = [(1.2, 0, 0.2, 0.0), (2.2, 1, 0.4, 1.0), (3.2, 0, 0.6, -1.0), (4.2, 1, 0.8, 2.0), (5.2, 0, 0.3, -2.0), (6.2, 1, 0.5, 4.0), (7.2, 0, 0.7, -4.0), (8.2, 1, 0.9, 6.0), (9.2, 0, 0.4, -6.0), (10.2, 1, 0.6, 8.0)]  # 定义十条测试样本用于覆盖评估。
training = make_records("train", training_specs)  # 构造只用于拟合点模型的训练集。
calibration = make_records("cal", calibration_specs)  # 构造只用于估计 q 的校准集。
test = make_records("test", test_specs)  # 构造最终不可参与校准的测试集。
print("教学实验输入：配送 ETA 三段数据")  # 标记下方为确定性脱敏样本。
print("split counts=", {"train": len(training), "calibration": len(calibration), "test": len(test)})  # 展示三段数据规模。
print("测试样本预览")  # 标记下表展示真实业务字段。
print("样本       distance rain load  actual  hidden_noise")  # 输出测试输入表头。
for record in test:  # 逐条展示最终评测样本。
    print(f"{record['id']:<10} {record['distance_km']:>8.1f} {record['rain']:>4} {record['courier_load']:>4.1f} {record['actual_minutes']:>7.2f} {record['noise']:>12.1f}")  # 输出当前配送特征和实际 ETA。

教学实验输入：配送 ETA 三段数据
split counts= {'train': 12, 'calibration': 10, 'test': 10}
测试样本预览
样本       distance rain load  actual  hidden_noise
test-01         1.2    0  0.2   18.20          0.0
test-02         2.2    1  0.4   29.00          1.0
test-03         3.2    0  0.6   22.80         -1.0
test-04         4.2    1  0.8   35.60          2.0
test-05         5.2    0  0.3   24.60         -2.0
test-06         6.2    1  0.5   40.40          4.0
test-07         7.2    0  0.7   28.20         -4.0
test-08         8.2    1  0.9   48.00          6.0
test-09         9.2    0  0.4   29.00         -6.0
test-10        10.2    1  0.6   52.80          8.0


## 2. Baseline / 基线：线性点预测加固定 ±2 分钟

先用正规方程手写拟合 ETA 点模型。基线无论距离、天气和校准误差都使用固定两分钟半宽，直接在十条测试上计算覆盖。

In [2]:
def design_matrix(records):  # 把业务记录转换为 bias、距离、雨天和负载特征矩阵。
    return np.array([[1.0, record["distance_km"], float(record["rain"]), record["courier_load"]] for record in records], dtype=np.float64)  # 返回样本乘四的数值矩阵。
def target_vector(records):  # 提取实际 ETA 目标向量。
    return np.array([record["actual_minutes"] for record in records], dtype=np.float64)  # 返回浮点分钟数。
training_matrix = design_matrix(training)  # 构造训练特征矩阵。
training_targets = target_vector(training)  # 构造训练目标。
ridge = np.eye(training_matrix.shape[1], dtype=np.float64) * 1.0e-8  # 加入极小对角正则保证正规方程稳定。
coefficients = np.linalg.solve(training_matrix.T @ training_matrix + ridge, training_matrix.T @ training_targets)  # 手写求解线性回归参数。
test_predictions = design_matrix(test) @ coefficients  # 对十条测试计算点预测。
baseline_half_width = 2.0  # 设定没有统计依据的固定区间半宽。
baseline_rows = []  # 保存逐测试样本固定区间结果。
for record, prediction in zip(test, test_predictions):  # 逐样本构造固定宽度区间。
    lower = prediction - baseline_half_width  # 计算固定区间下界。
    upper = prediction + baseline_half_width  # 计算固定区间上界。
    covered = lower <= record["actual_minutes"] <= upper  # 检查实际 ETA 是否落在区间内。
    baseline_rows.append({"id": record["id"], "prediction": float(prediction), "lower": float(lower), "upper": float(upper), "actual": record["actual_minutes"], "covered": covered})  # 保存当前点预测和覆盖结果。
baseline_coverage = sum(row["covered"] for row in baseline_rows) / len(baseline_rows)  # 计算固定区间经验覆盖率。
print("Point model coefficients [bias,distance,rain,load]=", np.round(coefficients, 5).tolist())  # 展示手写模型学到的业务系数。
print("Baseline固定区间")  # 标记下表展示固定宽度失败样本。
print("样本       prediction  interval             actual  covered")  # 输出基线结果表头。
for row in baseline_rows:  # 逐样本展示固定区间。
    print(f"{row['id']:<10} {row['prediction']:>10.2f} [{row['lower']:>6.2f},{row['upper']:>6.2f}] {row['actual']:>8.2f} {str(row['covered']):>8}")  # 输出当前测试覆盖。
print(f"Baseline coverage={baseline_coverage:.1%}，平均宽度={2.0 * baseline_half_width:.1f}分钟")  # 汇总固定区间质量。

Point model coefficients [bias,distance,rain,load]= [14.85642, 2.02587, 6.92077, 4.03194]
Baseline固定区间
样本       prediction  interval             actual  covered
test-01         18.09 [ 16.09, 20.09]    18.20     True
test-02         27.85 [ 25.85, 29.85]    29.00     True
test-03         23.76 [ 21.76, 25.76]    22.80     True
test-04         33.51 [ 31.51, 35.51]    35.60    False
test-05         26.60 [ 24.60, 28.60]    24.60    False
test-06         36.35 [ 34.35, 38.35]    40.40    False
test-07         32.27 [ 30.27, 34.27]    28.20    False
test-08         42.02 [ 40.02, 44.02]    48.00    False
test-09         35.11 [ 33.11, 37.11]    29.00    False
test-10         44.86 [ 42.86, 46.86]    52.80    False
Baseline coverage=30.0%，平均宽度=4.0分钟


## 3. 底层实现：独立校准残差与有限样本顺序统计量

目标误覆盖率 alpha=0.2。对十个校准绝对残差排序，rank=`ceil((n+1)*(1-alpha))=9`，取第九小残差作为 q；不使用插值。

In [3]:
alpha = 0.20  # 设定目标边际覆盖率为百分之八十。
calibration_predictions = design_matrix(calibration) @ coefficients  # 用冻结点模型预测独立校准集。
calibration_targets = target_vector(calibration)  # 读取校准实际 ETA。
calibration_scores = np.abs(calibration_targets - calibration_predictions)  # 计算 absolute residual nonconformity score。
ordered_scores = np.sort(calibration_scores)  # 从小到大排列十个校准分数。
finite_sample_rank = math.ceil((len(calibration_scores) + 1) * (1.0 - alpha))  # 计算有限样本保守一基 rank。
finite_sample_rank = min(max(finite_sample_rank, 1), len(calibration_scores))  # 将极端 alpha 的 rank 限制在有效范围。
conformal_q = float(ordered_scores[finite_sample_rank - 1])  # 取得不插值的保守校准半宽。
print("校准残差中间量")  # 标记下表展示每条校准样本而非只输出 q。
print("样本       prediction  actual  abs_residual")  # 输出校准结果表头。
for record, prediction, score in zip(calibration, calibration_predictions, calibration_scores):  # 逐条展示独立校准误差。
    print(f"{record['id']:<10} {prediction:>10.3f} {record['actual_minutes']:>7.3f} {score:>12.3f}")  # 输出当前校准 nonconformity score。
print("ordered_scores=", np.round(ordered_scores, 4).tolist())  # 展示顺序统计量输入。
print(f"n={len(calibration_scores)}，alpha={alpha}，rank={finite_sample_rank}，q={conformal_q:.4f}")  # 展示有限样本分位数计算结果。

校准残差中间量
样本       prediction  actual  abs_residual
cal-01         19.105  18.200        0.905
cal-02         28.858  30.000        1.142
cal-03         24.769  22.800        1.969
cal-04         31.700  33.800        2.100
cal-05         29.224  26.200        3.024
cal-06         36.558  39.600        3.042
cal-07         32.470  27.400        5.070
cal-08         42.626  47.600        4.974
cal-09         35.312  28.200        7.112
cal-10         45.871  52.800        6.929
ordered_scores= [0.9048, 1.1422, 1.9693, 2.1, 3.0243, 3.0419, 4.9742, 5.0696, 6.9288, 7.1118]
n=10，alpha=0.2，rank=9，q=6.9288


## 4. 逐样本结果与结果解读

用同一个点预测分别构造固定区间和 conformal 区间。覆盖只在测试集计算；校准集不能重复当成最终评测集。

In [4]:
corrected_rows = []  # 保存十条测试样本的 conformal 区间。
for record, prediction, baseline in zip(test, test_predictions, baseline_rows):  # 对同一测试点扩展校准半宽。
    lower = float(prediction - conformal_q)  # 计算 conformal 下界。
    upper = float(prediction + conformal_q)  # 计算 conformal 上界。
    covered = lower <= record["actual_minutes"] <= upper  # 检查测试实际 ETA 覆盖。
    corrected_rows.append({"id": record["id"], "prediction": float(prediction), "lower": lower, "upper": upper, "actual": record["actual_minutes"], "covered": covered, "baseline_covered": baseline["covered"], "rain": record["rain"]})  # 保存逐样本对照和分群字段。
corrected_coverage = sum(row["covered"] for row in corrected_rows) / len(corrected_rows)  # 计算 conformal 测试经验覆盖率。
dry_coverage = sum(row["covered"] for row in corrected_rows if row["rain"] == 0) / sum(row["rain"] == 0 for row in corrected_rows)  # 计算晴天分群覆盖率。
rain_coverage = sum(row["covered"] for row in corrected_rows if row["rain"] == 1) / sum(row["rain"] == 1 for row in corrected_rows)  # 计算雨天分群覆盖率。
print("样本       actual  fixed_interval/covered       conformal_interval/covered")  # 输出逐测试样本同数据对照表头。
for baseline, corrected in zip(baseline_rows, corrected_rows):  # 逐条比较固定和校准区间。
    print(f"{corrected['id']:<10} {corrected['actual']:>7.2f} [{baseline['lower']:>6.2f},{baseline['upper']:>6.2f}]/{str(baseline['covered']):<5}   [{corrected['lower']:>6.2f},{corrected['upper']:>6.2f}]/{str(corrected['covered']):<5}")  # 输出当前测试样本覆盖差异。
print(f"结果解读：固定±2分钟coverage={baseline_coverage:.1%}；Split Conformal coverage={corrected_coverage:.1%}、宽度={2.0 * conformal_q:.2f}分钟；晴天/雨天={dry_coverage:.1%}/{rain_coverage:.1%}。")  # 解释边际覆盖与区间宽度权衡。

样本       actual  fixed_interval/covered       conformal_interval/covered
test-01      18.20 [ 16.09, 20.09]/True    [ 11.17, 25.02]/True 
test-02      29.00 [ 25.85, 29.85]/True    [ 20.92, 34.78]/True 
test-03      22.80 [ 21.76, 25.76]/True    [ 16.83, 30.69]/True 
test-04      35.60 [ 31.51, 35.51]/False   [ 26.58, 40.44]/True 
test-05      24.60 [ 24.60, 28.60]/False   [ 19.67, 33.53]/True 
test-06      40.40 [ 34.35, 38.35]/False   [ 29.42, 43.28]/True 
test-07      28.20 [ 30.27, 34.27]/False   [ 25.34, 39.19]/True 
test-08      48.00 [ 40.02, 44.02]/False   [ 35.09, 48.95]/True 
test-09      29.00 [ 33.11, 37.11]/False   [ 28.18, 42.04]/True 
test-10      52.80 [ 42.86, 46.86]/False   [ 37.93, 51.79]/False
结果解读：固定±2分钟coverage=30.0%；Split Conformal coverage=90.0%、宽度=13.86分钟；晴天/雨天=100.0%/80.0%。


## 5. 失败案例与修正：用训练残差做校准

点模型已经直接拟合训练样本，训练残差系统性偏小。错误地从训练集取 q 会在同一测试集产生窄区间和低覆盖；独立 calibration 恢复更可靠的宽度。

In [5]:
training_predictions = training_matrix @ coefficients  # 计算模型见过的训练样本预测。
training_scores = np.abs(training_targets - training_predictions)  # 计算有乐观偏差的训练残差。
training_rank = math.ceil((len(training_scores) + 1) * (1.0 - alpha))  # 按同一公式计算错误校准 rank。
training_rank = min(max(training_rank, 1), len(training_scores))  # 把 rank 限制在训练残差范围。
leaked_q = float(np.sort(training_scores)[training_rank - 1])  # 从模型见过的数据取得过窄半宽。
leaked_coverage = sum(prediction - leaked_q <= record["actual_minutes"] <= prediction + leaked_q for record, prediction in zip(test, test_predictions)) / len(test)  # 在测试集评估训练残差区间。
proper_q = conformal_q  # 读取独立校准集得到的正确半宽。
proper_coverage = corrected_coverage  # 读取独立校准区间测试覆盖。
print(f"错误行为：training-residual q={leaked_q:.4f}，test coverage={leaked_coverage:.1%}，训练数据泄漏使区间过窄。")  # 展示用训练残差校准的实际失败指标。
print(f"修正行为：holdout-calibration q={proper_q:.4f}，test coverage={proper_coverage:.1%}，目标边际覆盖={1.0-alpha:.1%}。")  # 展示独立校准后的宽度和覆盖。

错误行为：training-residual q=0.9820，test coverage=20.0%，训练数据泄漏使区间过窄。
修正行为：holdout-calibration q=6.9288，test coverage=90.0%，目标边际覆盖=80.0%。


## 6. 生产边界

小样本观测覆盖会波动，conformal 保证依赖校准和未来样本可交换。生产要监控分布漂移、按区域/雨天分群覆盖、滚动校准窗口、标签延迟、区间宽度 SLO、非负 ETA 截断、异常宽区间 abstention，并冻结训练/校准边界防止泄漏。

In [6]:
conformal_diagnostics = {"training_samples": len(training), "calibration_samples": len(calibration), "test_samples": len(test), "alpha": alpha, "finite_sample_rank": finite_sample_rank, "q": conformal_q, "baseline_coverage": baseline_coverage, "conformal_coverage": corrected_coverage, "rain_coverage": rain_coverage, "training_leakage_coverage": leaked_coverage, "exchangeability_verified": False}  # 汇总数据切分、覆盖和假设边界。
print("生产监控快照：", conformal_diagnostics)  # 输出不确定性服务应持续跟踪的指标。

生产监控快照： {'training_samples': 12, 'calibration_samples': 10, 'test_samples': 10, 'alpha': 0.2, 'finite_sample_rank': 9, 'q': 6.928812595400608, 'baseline_coverage': 0.3, 'conformal_coverage': 0.9, 'rain_coverage': 0.8, 'training_leakage_coverage': 0.2, 'exchangeability_verified': False}


## 7. 最小回归测试

断言覆盖三段数据、有限样本 rank、同数据覆盖改善、边际目标和训练泄漏失败。

In [7]:
assert len(training) >= 6 and len(calibration) >= 6 and len(test) >= 6  # 保证训练、校准和测试均有至少六条真实业务记录。
assert finite_sample_rank == math.ceil((len(calibration_scores) + 1) * (1.0 - alpha))  # 保证使用有限样本保守顺序统计量。
assert conformal_q > baseline_half_width and corrected_coverage > baseline_coverage  # 保证独立校准区间在同一测试集改善覆盖。
assert corrected_coverage >= 1.0 - alpha  # 保证受控测试的经验覆盖达到目标边际水平。
assert leaked_q < proper_q and leaked_coverage < proper_coverage  # 保证训练残差泄漏真实产生过窄区间和较低覆盖。
assert all(math.isfinite(row["lower"]) and math.isfinite(row["upper"]) and row["lower"] <= row["upper"] for row in corrected_rows)  # 保证全部输出区间有限且上下界有序。